# 003 Refactor a Skill with References

这是第三课：什么时候该把细节从 `SKILL.md` 下沉到 `references/`。

学习目标：

1. 理解为什么 Skill 不能一直把所有内容堆在 `SKILL.md` 里
2. 学会判断什么时候该引入 `references/`
3. 学会把“工作流”和“具体资料”拆开
4. 结合真实天气 Skill 做一次正式重构

这节课继续使用：

- `.agents/skills/weather-query-assistant/`


## 先明确这节课解决什么问题

第二课里，我们故意把天气 Skill 做得很小。

但随着这份 Skill 开始变真实，`SKILL.md` 里已经出现了越来越多的内容：

- `wttr.in` 查询方式
- 格式码
- 参数技巧
- Open-Meteo fallback

这些东西不是 workflow 本身，而是“查询资料”。

这时就该开始学 `references/` 了。


## 先看当前真实目录

这次不是新增一个抽象目录，而是直接看重构后的真实天气 Skill。


In [1]:
from pathlib import Path


skill_root = Path('.agents/skills/weather-query-assistant')
print('exists =', skill_root.exists())
print('path =', skill_root.resolve())


exists = True
path = /home/dev/bxc/fastapi-study/.agents/skills/weather-query-assistant


In [2]:
def print_tree(root: Path, prefix: str = '') -> None:
    entries = sorted(root.iterdir(), key=lambda p: (p.is_file(), p.name))
    for index, entry in enumerate(entries):
        connector = '└── ' if index == len(entries) - 1 else '├── '
        print(prefix + connector + entry.name)
        if entry.is_dir():
            next_prefix = prefix + ('    ' if index == len(entries) - 1 else '│   ')
            print_tree(entry, next_prefix)


print(skill_root)
print_tree(skill_root)


.agents/skills/weather-query-assistant
├── agents
│   └── openai.yaml
├── references
│   └── weather_sources.md
└── SKILL.md


## 先读重构后的 `SKILL.md`

这一步最关键。

你应该能观察到：

- `SKILL.md` 变短了
- workflow 还在
- 风格约束还在
- 但具体查询样例被拿走了

这正是一次健康的 Skill 重构。


In [3]:
print((skill_root / 'SKILL.md').read_text(encoding='utf-8'))


---
name: weather
description: Get current weather and forecasts (no API key required).
homepage: https://wttr.in/:help
metadata: {"nanobot":{"emoji":"🌤️","requires":{"bins":["curl"]}}}
---

# Weather

Use this skill when the user asks for current weather or a short forecast for a specific location.

## Workflow

1. Check whether the user gave a clear location.
2. If the location is missing or ambiguous, ask a short clarification question.
3. Use `wttr.in` as the primary source.
4. Use Open-Meteo as a fallback when JSON output or more programmatic structure is needed.
5. Answer concisely with practical details:
   - location
   - current condition or forecast summary
   - temperature
   - humidity or wind when relevant
   - rain risk when relevant
6. Do not guess when weather data is unavailable.

## References

- Read `references/weather_sources.md` for concrete `wttr.in` and Open-Meteo query patterns.

## Style

- Prefer short, direct answers.
- Use the exact place name from the user

## 再读 `references/weather_sources.md`

这里放的是“使用资料”，不是“触发工作流”。

也就是说：

- `SKILL.md` 负责告诉你要做什么
- `references/` 负责告诉你具体怎么查


In [4]:
print((skill_root / 'references' / 'weather_sources.md').read_text(encoding='utf-8'))


# Weather Sources

This reference contains the concrete query patterns for the weather skill.

Use `wttr.in` as the primary source for concise, human-readable weather output.

Use Open-Meteo as the fallback when a structured JSON response is more useful.

## wttr.in

Quick one-liner:

```bash
curl -s "wttr.in/London?format=3"
# Output: London: ⛅️ +8°C
```

Compact format:

```bash
curl -s "wttr.in/London?format=%l:+%c+%t+%h+%w"
# Output: London: ⛅️ +8°C 71% ↙5km/h
```

Full forecast:

```bash
curl -s "wttr.in/London?T"
```

Format codes:

- `%c` condition
- `%t` temp
- `%h` humidity
- `%w` wind
- `%l` location
- `%m` moon

Tips:

- URL-encode spaces: `wttr.in/New+York`
- Airport codes: `wttr.in/JFK`
- Units: `?m` for metric, `?u` for USCS
- Today only: `?1`
- Current only: `?0`
- PNG output: `curl -s "wttr.in/Berlin.png" -o /tmp/weather.png`

## Open-Meteo

Use this when a structured JSON response is more useful than text.

```bash
curl -s "https://api.open-meteo.com/v1/forecast?latitu

## 这次重构到底改进了什么

把细节拆出去，不是为了目录好看，而是为了让 Skill 更稳定。

现在这份天气 Skill 的分层更清楚了：

- `SKILL.md`：触发条件、workflow、回答风格
- `references/weather_sources.md`：外部服务查询资料

这种拆法会让后续维护轻松很多。


In [5]:
refactor_result = {
    'SKILL.md_keeps': ['trigger', 'workflow', 'style'],
    'references_keeps': ['wttr.in examples', 'format codes', 'URL tips', 'Open-Meteo fallback details'],
}

from pprint import pprint
pprint(refactor_result)


{'SKILL.md_keeps': ['trigger', 'workflow', 'style'],
 'references_keeps': ['wttr.in examples',
                      'format codes',
                      'URL tips',
                      'Open-Meteo fallback details']}


## 什么时候该引入 `references/`

你不需要死记规则，但可以记住下面几个信号。

当出现这些情况时，通常就该拆 `references/`：

1. `SKILL.md` 开始变长
2. 里面出现大量样例命令或资料表
3. 某部分内容不是 workflow，而是查阅资料
4. 某部分信息只有在少数任务里才会用到


In [6]:
when_to_add_references = [
    'SKILL.md is getting long',
    'too many concrete command examples',
    'content is reference material instead of workflow',
    'details are only needed in some tasks',
]

pprint(when_to_add_references)


['SKILL.md is getting long',
 'too many concrete command examples',
 'content is reference material instead of workflow',
 'details are only needed in some tasks']


## 什么时候还不该拆 `references/`

也不是所有 Skill 都要强行拆目录。

如果内容还很小，而且这些细节每次都会用到，那继续放在 `SKILL.md` 里也没问题。

关键不是“有没有 `references/`”，而是“拆完之后是否更清晰”。


In [7]:
when_not_to_add_references = [
    'workflow is still very short',
    'examples are very few',
    'all details are needed almost every time',
    'splitting would make navigation worse instead of better',
]

pprint(when_not_to_add_references)


['workflow is still very short',
 'examples are very few',
 'all details are needed almost every time',
 'splitting would make navigation worse instead of better']


## 这份天气 Skill 为什么现在适合拆

因为它已经不只是“回答天气问题”这么简单了。

它还承载了：

- `wttr.in` 的多种查询格式
- URL 编码技巧
- 单位切换
- Open-Meteo fallback

这些内容都是真资料，不适合继续挤在 `SKILL.md` 正文里。


In [8]:
why_weather_skill_now_needs_references = {
    'multiple_source_patterns': True,
    'many_concrete_examples': True,
    'query_tips_exist': True,
    'fallback_source_exists': True,
}

pprint(why_weather_skill_now_needs_references)


{'fallback_source_exists': True,
 'many_concrete_examples': True,
 'multiple_source_patterns': True,
 'query_tips_exist': True}


## 正式开发时应该怎么拆

一个很稳的做法是：

1. 先判断哪些内容是 workflow
2. 再判断哪些内容是查阅资料
3. workflow 留在 `SKILL.md`
4. 查阅资料挪到 `references/`
5. 在 `SKILL.md` 里留下清楚的引用入口

这比“按感觉拆文件”靠谱得多。


In [9]:
splitting_rule = {
    'keep_in_skill_md': ['trigger condition', 'workflow', 'style', 'resource entry points'],
    'move_to_references': ['detailed examples', 'parameter tables', 'fallback API details', 'lookup tips'],
}

pprint(splitting_rule)


{'keep_in_skill_md': ['trigger condition',
                      'workflow',
                      'style',
                      'resource entry points'],
 'move_to_references': ['detailed examples',
                        'parameter tables',
                        'fallback API details',
                        'lookup tips']}


## 这节课的正式开发价值

学完这一课，你应该开始形成一个实际习惯：

- Skill 先求可用
- 然后再做分层
- 分层时优先考虑维护成本

这才是正式开发里的 Skill 演进方式。


## 如果继续往下走，第四课最自然的方向是什么

现在我们已经有了：

- 最小 Skill
- `agents/openai.yaml`
- `references/`

下一步最自然就是：

- 加 `scripts/`

对这份天气 Skill 来说，一个很合适的第四课例子是：

- `normalize_location.py`

它可以用来把：

- `beijing`
- `Beijing`
- `New York`
- `new+york`

标准化成更稳定的查询输入。


In [10]:
lesson_four_candidate = {
    'script_name': 'normalize_location.py',
    'goal': 'normalize user-provided locations before querying wttr.in or Open-Meteo',
    'why_next': 'this is the first repeated deterministic action worth scripting',
}

pprint(lesson_four_candidate)


{'goal': 'normalize user-provided locations before querying wttr.in or '
         'Open-Meteo',
 'script_name': 'normalize_location.py',
 'why_next': 'this is the first repeated deterministic action worth scripting'}


## 当前阶段结论

你现在需要记住：

1. `references/` 不是默认必备，但当 Skill 资料开始变多时非常重要
2. `SKILL.md` 负责 workflow，`references/` 负责资料细节
3. 真正该拆的时候，不是因为“目录结构看起来更专业”，而是因为维护成本在上升
4. 这份天气 Skill 已经到了适合拆 `references/` 的阶段
5. Skill 的成长顺序通常是：最小版 -> 加 metadata -> 加 references -> 再考虑 scripts

下一步建议：

- 继续第四课：给天气 Skill 增加 `normalize_location.py`
